# DLT Workshop

In [2]:
import urllib.request

# Define the exact 2026 dlt workshop homework directory URL
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/workshops/dlt/homework"

# List of targets: (remote_filename, local_filename)
files_to_download = [
    ("agent.py", "agent.py"),
    ("ingest.py", "ingest.py"),
    ("main.py", "main.py"),
    (".env.example", ".env")  # Downloads and renames it directly
]

for remote_name, local_name in files_to_download:
    url = f"{PREFIX}/{remote_name}"
    try:
        urllib.request.urlretrieve(url, local_name)
        print(f"✅ {local_name} downloaded successfully!")
    except Exception as e:
        print(f"❌ Failed to download {remote_name}: {e}")

✅ agent.py downloaded successfully!
✅ ingest.py downloaded successfully!
✅ main.py downloaded successfully!
✅ .env downloaded successfully!


### Question 2. Load traces into DuckDB with dlt
Generate a read token for your Logfire project and set it as LOGFIRE_READ_TOKEN in .env.

Initialize a dlt-hub project like in the workshop. Then ask your coding agent to pull the data from Pydantic Logfire and save it into DuckDB.

The dltHub AI workbench has a ready-made context for Logfire. Point your agent to it: https://dlthub.com/context/source/logfire

If you don't currently use a coding agent, you can use something like OpenCode: you should be able to complete one session with the free account.

Alternatively, you can do it in the old way (using ChatGPT or your favorite search engine).

In [ ]:
from typing import Iterator
import dlt
from dlt.sources.helpers import requests
import duckdb

LOGFIRE_BASE_URLS = {
    "us": "https://logfire-us.pydantic.dev",
    "eu": "https://logfire-eu.pydantic.dev",
}

PAGE_LIMIT = 10_000
DEFAULT_START = "2024-01-01T00:00:00Z"

@dlt.source
def logfire_source(
    read_token: str = dlt.secrets.value,
    region: str = dlt.config.value,
) -> Iterator[dlt.sources.DltResource]:
    """Logfire source. Currently exposes a single resource: `spans`."""

    @dlt.resource(
        name="spans",
        write_disposition="append",
        primary_key="span_id",
    )
    def spans(
        start_timestamp: dlt.sources.incremental[str] = dlt.sources.incremental(
            "start_timestamp",
            initial_value=DEFAULT_START,
        ),
    ):
        base_url = LOGFIRE_BASE_URLS.get(region, LOGFIRE_BASE_URLS["us"])
        headers = {
            "Authorization": f"Bearer {read_token}",
            "Accept": "application/json",
        }

        min_ts = start_timestamp.last_value

        while True:
            params = {
                "sql": "SELECT * FROM records ORDER BY start_timestamp ASC",
                "row_oriented": "true",
                "limit": PAGE_LIMIT,
                "min_timestamp": min_ts,
            }
            response = requests.get(f"{base_url}/v1/query", params=params, headers=headers)
            response.raise_for_status()

            payload = response.json()
            
            # Handle row-oriented responses
            if isinstance(payload, list):
                rows = payload
            elif isinstance(payload, dict) and "rows" in payload:
                rows = payload["rows"]
            
            # Handle column-oriented responses
            elif isinstance(payload, dict) and "columns" in payload:
                columns = payload["columns"]
                names = [c["name"] for c in columns]
                values = [c["values"] for c in columns]
                rows = [
                    dict(zip(names, row_values))
                    for row_values in zip(*values)
                ]
            else:
                raise ValueError(f"Unexpected response format: {payload}")
            
            if not rows:
                break
            
            yield rows
            
            if len(rows) < PAGE_LIMIT:
                break
            
            min_ts = rows[-1]["start_timestamp"]

    return spans

def run() -> None:
    pipeline = dlt.pipeline(
        pipeline_name="logfire_pipeline",
        destination="duckdb",
        dataset_name="logfire_data",
    )

    # Hardcoding your verified token and region details directly
    direct_token = ""
    
    source = logfire_source(
        read_token=direct_token,
        region="us",
    )
    
    load_info = pipeline.run(source)
    print(load_info)

if __name__ == "__main__":
    # 1. Run the pipeline extraction
    run()

    # 2. Automatically execute the verification script to get the answer
    print("\n--- Verifying Homework Answer ---")
    conn = duckdb.connect("logfire_pipeline.duckdb")
    result = conn.execute("""
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_schema = 'logfire_data';
    """).fetchall()

    print(f"Total tables created by dlt: {result[0][0]}")

Pipeline logfire_pipeline load step completed in ---
0 load package(s) were loaded to destination duckdb and into dataset None
The duckdb destination used duckdb:///c:\Users\MatenTech\llm-zoomcamp-2026-code\6-dlt_workshop\logfire_pipeline.duckdb location to store data

--- Verifying Homework Answer ---
Total tables created by dlt: 18


### Question 3. Query traces with an agent

Using a coding agent (you can also write the code by hand) find the input token usage for the agent run from Q1.

The token counts are stored in the span attributes as gen_ai.usage.input_tokens. Sum them across all LLM calls within the trace. The number depends on how many searches the agent made, so report the range it falls into:

100 - 500
1500 - 5000
10000 - 20000
50000 - 100000

In [14]:
import duckdb

# Connect to the DuckDB instance built during Question 2
conn = duckdb.connect("logfire_pipeline.duckdb")

# Use the precise column binding identified by the error
query = """
SELECT SUM(CAST(attributes__gen_ai_usage_input_tokens AS INT)) 
FROM logfire_data.spans
WHERE attributes__gen_ai_usage_input_tokens IS NOT NULL;
"""

result = conn.execute(query).fetchone()[0]
print(f"Your exact total input tokens: {result}")

Your exact total input tokens: 19372
